### Dynamic Hedging with Bond Portfolio

We want to engage in a 3-year investment with a return goal of $100,000

Assume interest rate at t = 0 is 12%

Assume there are only 2 products: 
- Bond A: 5-year coupon bond with annual coupon C = $10 and face value F = $100
- Bond B: 1-year zero-coupon bond with face value F = $100

In [111]:
import numpy as np
import matplotlib.pyplot as plt

def price_bond(CFs, r):
    return round(np.sum([CF * np.exp(-(t +1) * r) for t,CF in enumerate(CFs)]), 3)

def duration_coupon_bond(CFs, r):
    d = 1/price_bond(CFs, r)
    d *= np.sum([(t+1) * CF * np.exp(-(t +1) * r) for t,CF in enumerate(CFs)])
    return round(d, 3)

# General bond facts
C = 10
F = 100
r0 = 0.12

GENERAL CONSIDERATIONS: 
- Bond A is a coupon bond (5-year), this means each year a coupon is delivered, the duration changes over time as we approach maturity
- Bond B is a zero-coupon (1-year), this means each year at maturity we obtain the face value, the duration is always 1 year
- Our strategy will need to be adapted as we proceed toward the end of our investment timeline and will need to consider the value of the interest rate

Note that the portfolio can be build using the following structure

$$\begin{align}
    P &= a \cdot P_A(y(0)) + b \cdot P_B(y(0)) \\
    D &= \omega_A D_A(y(0)) + \omega_B D_B(y(0)) \\
    \omega_A &= a \frac{P_A(y(0))}{P} \\
    \omega_B &= b \frac{P_B(y(0))}{P} 
\end{align}$$

By solving a systme of linear equation we find

$$
\begin{pmatrix}
1 \\
D
\end{pmatrix}
=
\begin{pmatrix}
1 & 1 \\
D_A & D_B
\end{pmatrix}
\begin{pmatrix}
w_A \\
w_B
\end{pmatrix}
$$

$$
\begin{pmatrix}
w_A \\
w_B
\end{pmatrix}
=
\begin{pmatrix}
1 & 1 \\
D_A & D_B
\end{pmatrix}^{-1}
\begin{pmatrix}
1 \\
D
\end{pmatrix}

$$

In [112]:
# VALUE OF PORTFOLIO AND BOND PRICES
# Initial investment, given r = 0.12, we invest present day value of $100,000 to be collected in 3 years
P_t0 = 100_000 * np.exp(-3 * r0)
print(f"The initial investment is given by {round(P_t0, 2)}")

Pa_t0 = price_bond([C, C, C, C, F+C], r=r0)
Pb_t0 = price_bond([F], r=r0)
print(f"The price of bond A at t = 0 is ${Pa_t0}")
print(f"The price of bond B at t = 0 is ${Pb_t0}")

# BOND DURATIONS
# Find Duration numerically (finite first order derivative)
h = 1e-4
Da_t0_num = -1/Pa_t0 * (price_bond([C, C, C, C, F+C], r0 + h) - price_bond([C, C, C, C, F+C], r0 - h)) / (2 * h)
Db_t0_num = -1/Pa_t0 * (price_bond([F], r0 + h) - price_bond([F], r0 - h)) / (2 * h)

Da_t0 = duration_coupon_bond([C, C, C, C, F+C], r=r0)
Db_t0 = 1
print(f"The Duration of bond A at t = 0 is {Da_t0} [{round(Da_t0_num, 3)}]")
print(f"The Duration of bond B at t = 0 is {Db_t0} [{round(Db_t0_num, 3)}]")


# BUILDING PORTFOLIO
# We need to build a bond portfolio with Duration D = 3 to make it stable to changes in interest rate (plan is to invest for 3 years)
# Solve to find wA and wB and then weights a and b
D = 3

y = np.array([1, D])
A = np.array([[1, 1],
              [Da_t0, Db_t0]])
w0 = np.linalg.solve(A, y)
print(f"The weights to achieve D = 3 are: wA = {round(w0[0], 4)}, wB = {round(w0[1], 4)}")

a0 = round(w0[0] * P_t0 / Pa_t0, 4)
b0 = round(w0[1] * P_t0 / Pb_t0, 4)
print(f"""The portfolio (value = ${round(Pa_t0 * a0 + Pb_t0 * b0, 2)}) is built with:
        * {a0} bonds of type A for a value of ${round(Pa_t0 * a0, 2)}
        * {b0} bonds of type B for a value of ${round(Pb_t0 * b0, 2)}""")

The initial investment is given by 69767.63
The price of bond A at t = 0 is $90.269
The price of bond B at t = 0 is $88.692
The Duration of bond A at t = 0 is 4.122 [4.154]
The Duration of bond B at t = 0 is 1 [0.997]
The weights to achieve D = 3 are: wA = 0.6406, wB = 0.3594
The portfolio (value = $69767.64) is built with:
        * 495.1223 bonds of type A for a value of $44694.19
        * 282.7024 bonds of type B for a value of $25073.44


### How will our new portfolio (designed to be immune to interest rate changes) behave?

#### SCENARIO 1: 
At t = 1, the interest rate increases to 14%:
- First coupon is due for a total of $10 x 495.122 = $4,951.22
- Zero-coupon bond matures for a total of $100 x 282.7024 = $28,270.24
- Bond A has a new value and new duration after interest rate change and 1 less year in its lifetime


In [113]:
r1 = 0.14

# UPDATE PRICE
# Bond A new value becomes (only 4 years are left with their associated cash flows and new interest rate)
Pa_t1 = price_bond([C, C, C, F+C], r=r1)
Da_t1 = duration_coupon_bond([C, C, C, F+C], r=r1)
# Bond B new value becomes (due to change in interest rate)
Pb_t1 = price_bond([F], r=r1)
Db_t1 = 1

print(f"Bond A value after interest rate change: ${Pa_t1}")
print(f"Bond A duration after interest rate change: {Da_t1}")

print(f"Bond B value after interest rate change: ${Pb_t1}")
print(f"Bond B duration: {Db_t1}")

P_t1 = C * a0 + F * b0 + Pa_t1 * a0
print(f"At this point the portfoltio is valued: ${round(P_t1, 2)}")


# UPDATE PORTFOLIO
# We need to adapt for a Duration D = 2 to make it stable to changes in interest rate (plan is to invest for 2 years)
# Solve to find wA and wB for building portfolio with Duration = 2
D = 2

y = np.array([1, D])
A = np.array([[1, 1],
              [Da_t1, Db_t1]])
w1= np.linalg.solve(A, y)
print(f"The weights to achieve D = 2 are: wA = {round(w1[0], 4)}, wB = {round(w1[1], 4)}")

a1 = round(w1[0] * P_t1 / Pa_t1, 4)
b1 = round(w1[1] * P_t1 / Pb_t1, 4)
print(f"""The portfolio (value = ${round(Pa_t1 * a1 + Pb_t1 * b1, 2)}) is built with:
        * {a1} bonds of type A for a value of ${round(Pa_t1 * a1, 2)}
        * {b1} bonds of type B for a value of ${round(Pb_t1 * b1, 2)}
        
    Note that to adapt our portfolio at t = 1 we need to: 
        * sell {round(a0-a1, 4)} bonds of type A at the new price ${Pa_t1}
        * buy {b1} bonds of type B at the new price ${Pb_t1}""")

Bond A value after interest rate change: $85.655
Bond A duration after interest rate change: 3.442
Bond B value after interest rate change: $86.936
Bond B duration: 1
At this point the portfoltio is valued: $75631.16
The weights to achieve D = 2 are: wA = 0.4095, wB = 0.5905
The portfolio (value = $75631.16) is built with:
        * 361.5783 bonds of type A for a value of $30970.99
        * 513.7132 bonds of type B for a value of $44660.17

    Note that to adapt our portfolio at t = 1 we need to: 
        * sell 133.544 bonds of type A at the new price $85.655
        * buy 513.7132 bonds of type B at the new price $86.936


(SCENARIO 1.1) At t = 2, the interest rate decreases to 9%:
- Second coupon is due for a total of $10 x 361.5783 = $3,615.78
- Zero-coupon bond matures for a total of $100 x 513.7132 = $51,371.32
- Bond A has a new value and new duration after interest rate change and 1 less year in its lifetime

In [114]:
r2_1 = 0.09

# UPDATE PRICE
# Bond A new value becomes (only 3 years are left with their associated cash flows and new interest rate)
Pa_t2_1 = price_bond([C, C, F+C], r=r2_1)
Da_t2_1 = duration_coupon_bond([C, C, F+C], r=r2_1)
# Bond B new value becomes (due to change in interest rate)
Pb_t2_1 = price_bond([F], r=r2_1)
Db_t2_1 = 1

print(f"Bond A value after interest rate change: ${Pa_t2_1}")
print(f"Bond A duration after interest rate change: {Da_t2_1}")

print(f"Bond B value after interest rate change: ${Pb_t2_1}")
print(f"Bond B duration: {Db_t2_1}")

P_t2_a = C * a1 + F * b1 + Pa_t2_1 * a1
print(f"At this point the portfoltio is valued: ${round(P_t2_a, 2)}")

# UPDATE PORTFOLIO
# We need to adapt for a Duration D = 1 to make it stable to changes in interest rate (plan is to invest for 1 year)
# Solve to find wA and wB for building portfolio with Duration = 1
D = 1

y = np.array([1, D])
A = np.array([[1, 1],
              [Da_t2_1, Db_t2_1]])
w2_1 = np.linalg.solve(A, y)
print(f"The weights to achieve D = 1 are: wA = {round(w2_1[0], 4)}, wB = {round(w2_1[1], 4)}")

a2_1 = round(w2_1[0] * P_t2_a / Pa_t2_1, 4)
b2_1 = round(w2_1[1] * P_t2_a / Pb_t2_1, 4)
print(f"""The portfolio (value = ${round(Pa_t2_1 * a2_1 + Pb_t2_1 * b2_1, 2)}) is built with:
        * {a2_1} bonds of type A for a value of ${round(Pa_t2_1 * a2_1, 2)}
        * {b2_1} bonds of type B for a value of ${round(Pb_t2_1 * b2_1, 2)}
        
    Note that to adapt our portfolio at t = 1 we need to: 
        * sell {round(a1-a2_1, 4)} bonds of type A at the new price ${Pa_t2_1}
        * buy {b2_1} bonds of type B at the new price ${Pb_t2_1}""")

Bond A value after interest rate change: $101.464
Bond A duration after interest rate change: 2.738
Bond B value after interest rate change: $91.393
Bond B duration: 1
At this point the portfoltio is valued: $91674.28
The weights to achieve D = 1 are: wA = 0.0, wB = 1.0
The portfolio (value = $91674.28) is built with:
        * 0.0 bonds of type A for a value of $0.0
        * 1003.0777 bonds of type B for a value of $91674.28

    Note that to adapt our portfolio at t = 1 we need to: 
        * sell 361.5783 bonds of type A at the new price $101.464
        * buy 1003.0777 bonds of type B at the new price $91.393


Because the duration left in our investment is exactly 1 year, we sell all bonds of type A and only purchase bonds of type B with our money

This will secure a portfolio with D = 1 to fit our strategy

The final value will be given by the maturity of bonds of type B: $100 * 1003.0777 = $100,307.77

(SCENARIO 1.2) At t = 2, the interest rate increases again to 16%:
- Second coupon is due for a total of $10 x 361.5783 = $3,615.78
- Zero-coupon bond matures for a total of $100 x 513.7132 = $51,371.32
- Bond A has a new value and new duration after interest rate change and 1 less year in its lifetime

In [115]:
r2_2 = 0.16

# UPDATE PRICE
# Bond A new value becomes (only 3 years are left with their associated cash flows and new interest rate)
Pa_t2_2 = price_bond([C, C, F+C], r=r2_2)
Da_t2_2 = duration_coupon_bond([C, C, F+C], r=r2_2)
# Bond B new value becomes (due to change in interest rate)
Pb_t2_2 = price_bond([F], r=r2_2)
Db_t2_2 = 1

print(f"Bond A value after interest rate change: ${Pa_t2_2}")
print(f"Bond A duration after interest rate change: {Da_t2_2}")

print(f"Bond B value after interest rate change: ${Pb_t2_2}")
print(f"Bond B duration: {Db_t2_2}")

P_t2_2 = C * a1 + F * b1 + Pa_t2_2 * a1
print(f"At this point the portfoltio is valued: ${round(P_t2_2, 2)}")

# UPDATE PORTFOLIO
# We need to adapt for a Duration D = 1 to make it stable to changes in interest rate (plan is to invest for 1 year)
# Solve to find wA and wB for building portfolio with Duration = 1
D = 1

y = np.array([1, D])
A = np.array([[1, 1],
              [Da_t2_2, Db_t2_2]])
w2_2 = np.linalg.solve(A, y)
print(f"The weights to achieve D = 1 are: wA = {round(w2_2[0], 4)}, wB = {round(w2_2[1], 4)}")

a2_2 = round(w2_2[0] * P_t2_2 / Pa_t2_2, 4)
b2_2 = round(w2_2[1] * P_t2_2 / Pb_t2_2, 4)
print(f"""The portfolio (value = ${round(Pa_t2_2 * a2_2 + Pb_t2_2 * b2_2, 2)}) is built with:
        * {a2_2} bonds of type A for a value of ${round(Pa_t2_2 * a2_2, 2)}
        * {b2_2} bonds of type B for a value of ${round(Pb_t2_2 * b2_2, 2)}
        
    Note that to adapt our portfolio at t = 1 we need to: 
        * sell {round(a1-a2_2, 4)} bonds of type A at the new price ${Pa_t2_2}
        * buy {b2_2} bonds of type B at the new price ${Pb_t2_2}""")

Bond A value after interest rate change: $83.849
Bond A duration after interest rate change: 2.71
Bond B value after interest rate change: $85.214
Bond B duration: 1
At this point the portfoltio is valued: $85305.08
The weights to achieve D = 1 are: wA = 0.0, wB = 1.0
The portfolio (value = $85305.09) is built with:
        * 0.0 bonds of type A for a value of $0.0
        * 1001.0689 bonds of type B for a value of $85305.09

    Note that to adapt our portfolio at t = 1 we need to: 
        * sell 361.5783 bonds of type A at the new price $83.849
        * buy 1001.0689 bonds of type B at the new price $85.214


Because the duration left in our investment is exactly 1 year, we sell all bonds of type A and only purchase bonds of type B with our money

This will secure a portfolio with D = 1 to fit our strategy

The final value will be given by the maturity of bonds of type B: $100 * 1001.0689 = $100,106.89

#### SCENARIO 2: 
At t = 1, the interest rate decreases to 9%:
- First coupon is due for a total of $10 x 495.122 = $4,951.22
- Zero-coupon bond matures for a total of $100 x 282.7024 = $28,270.24
- Bond A has a new value and new duration after interest rate change and 1 less year in its lifetime


In [116]:
r1 = 0.09

# UPDATE PRICE
# Bond A new value becomes (only 4 years are left with their associated cash flows and new interest rate)
Pa_t1 = price_bond([C, C, C, F+C], r=r1)
Da_t1 = duration_coupon_bond([C, C, C, F+C], r=r1)
# Bond B new value becomes (due to change in interest rate)
Pb_t1 = price_bond([F], r=r1)
Db_t1 = 1

print(f"Bond A value after interest rate change: ${Pa_t1}")
print(f"Bond A duration after interest rate change: {Da_t1}")

print(f"Bond B value after interest rate change: ${Pb_t1}")
print(f"Bond B duration: {Db_t1}")

P_t1 = C * a0 + F * b0 + Pa_t1 * a0
print(f"At this point the portfoltio is valued: ${round(P_t1, 2)}")


# UPDATE PORTFOLIO
# We need to adapt for a Duration D = 2 to make it stable to changes in interest rate (plan is to invest for 2 years)
# Solve to find wA and wB for building portfolio with Duration = 2
D = 2

y = np.array([1, D])
A = np.array([[1, 1],
              [Da_t1, Db_t1]])
w1= np.linalg.solve(A, y)
print(f"The weights to achieve D = 2 are: wA = {round(w1[0], 4)}, wB = {round(w1[1], 4)}")

a1 = round(w1[0] * P_t1 / Pa_t1, 4)
b1 = round(w1[1] * P_t1 / Pb_t1, 4)
print(f"""The portfolio (value = ${round(Pa_t1 * a1 + Pb_t1 * b1, 2)}) is built with:
        * {a1} bonds of type A for a value of ${round(Pa_t1 * a1, 2)}
        * {b1} bonds of type B for a value of ${round(Pb_t1 * b1, 2)}
        
    Note that to adapt our portfolio at t = 1 we need to: 
        * sell {round(a0-a1, 4)} bonds of type A at the new price ${Pa_t1}
        * buy {b1} bonds of type B at the new price ${Pb_t1}""")

Bond A value after interest rate change: $101.87
Bond A duration after interest rate change: 3.492
Bond B value after interest rate change: $91.393
Bond B duration: 1
At this point the portfoltio is valued: $83659.57
The weights to achieve D = 2 are: wA = 0.4013, wB = 0.5987
The portfolio (value = $83659.58) is built with:
        * 329.55 bonds of type A for a value of $33571.26
        * 548.0542 bonds of type B for a value of $50088.32

    Note that to adapt our portfolio at t = 1 we need to: 
        * sell 165.5723 bonds of type A at the new price $101.87
        * buy 548.0542 bonds of type B at the new price $91.393


(SCENARIO 1.1) At t = 2, the interest rate increases to 14%:
- Second coupon is due for a total of $10 x 329.55 = $3,295.50
- Zero-coupon bond matures for a total of $100 x 548.0542 = $54,805.42
- Bond A has a new value and new duration after interest rate change and 1 less year in its lifetime

In [117]:
r2_1 = 0.14

# UPDATE PRICE
# Bond A new value becomes (only 3 years are left with their associated cash flows and new interest rate)
Pa_t2_1 = price_bond([C, C, F+C], r=r2_1)
Da_t2_1 = duration_coupon_bond([C, C, F+C], r=r2_1)
# Bond B new value becomes (due to change in interest rate)
Pb_t2_1 = price_bond([F], r=r2_1)
Db_t2_1 = 1

print(f"Bond A value after interest rate change: ${Pa_t2_1}")
print(f"Bond A duration after interest rate change: {Da_t2_1}")

print(f"Bond B value after interest rate change: ${Pb_t2_1}")
print(f"Bond B duration: {Db_t2_1}")

P_t2_a = C * a1 + F * b1 + Pa_t2_1 * a1
print(f"At this point the portfoltio is valued: ${round(P_t2_a, 2)}")

# UPDATE PORTFOLIO
# We need to adapt for a Duration D = 1 to make it stable to changes in interest rate (plan is to invest for 1 year)
# Solve to find wA and wB for building portfolio with Duration = 1
D = 1

y = np.array([1, D])
A = np.array([[1, 1],
              [Da_t2_1, Db_t2_1]])
w2_1 = np.linalg.solve(A, y)
print(f"The weights to achieve D = 1 are: wA = {round(w2_1[0], 4)}, wB = {round(w2_1[1], 4)}")

a2_1 = round(w2_1[0] * P_t2_a / Pa_t2_1, 4)
b2_1 = round(w2_1[1] * P_t2_a / Pb_t2_1, 4)
print(f"""The portfolio (value = ${round(Pa_t2_1 * a2_1 + Pb_t2_1 * b2_1, 2)}) is built with:
        * {a2_1} bonds of type A for a value of ${round(Pa_t2_1 * a2_1, 2)}
        * {b2_1} bonds of type B for a value of ${round(Pb_t2_1 * b2_1, 2)}
        
    Note that to adapt our portfolio at t = 1 we need to: 
        * sell {round(a1-a2_1, 4)} bonds of type A at the new price ${Pa_t2_1}
        * buy {b2_1} bonds of type B at the new price ${Pb_t2_1}""")

Bond A value after interest rate change: $88.527
Bond A duration after interest rate change: 2.718
Bond B value after interest rate change: $86.936
Bond B duration: 1
At this point the portfoltio is valued: $87274.99
The weights to achieve D = 1 are: wA = 0.0, wB = 1.0
The portfolio (value = $87274.99) is built with:
        * 0.0 bonds of type A for a value of $0.0
        * 1003.8993 bonds of type B for a value of $87274.99

    Note that to adapt our portfolio at t = 1 we need to: 
        * sell 329.55 bonds of type A at the new price $88.527
        * buy 1003.8993 bonds of type B at the new price $86.936


Because the duration left in our investment is exactly 1 year, we sell all bonds of type A and only purchase bonds of type B with our money

This will secure a portfolio with D = 1 to fit our strategy

The final value will be given by the maturity of bonds of type B: $100 * 1003.8993 = $100,389.93

(SCENARIO 1.2) At t = 2, the interest rate decreases again to 6%:
- Second coupon is due for a total of $10 x 329.55 = $3,295.50
- Zero-coupon bond matures for a total of $100 x 548.0542 = $54,805.42
- Bond A has a new value and new duration after interest rate change and 1 less year in its lifetime

In [118]:
r2_2 = 0.06

# UPDATE PRICE
# Bond A new value becomes (only 3 years are left with their associated cash flows and new interest rate)
Pa_t2_2 = price_bond([C, C, F+C], r=r2_2)
Da_t2_2 = duration_coupon_bond([C, C, F+C], r=r2_2)
# Bond B new value becomes (due to change in interest rate)
Pb_t2_2 = price_bond([F], r=r2_2)
Db_t2_2 = 1

print(f"Bond A value after interest rate change: ${Pa_t2_2}")
print(f"Bond A duration after interest rate change: {Da_t2_2}")

print(f"Bond B value after interest rate change: ${Pb_t2_2}")
print(f"Bond B duration: {Db_t2_2}")

P_t2_2 = C * a1 + F * b1 + Pa_t2_2 * a1
print(f"At this point the portfoltio is valued: ${round(P_t2_2, 2)}")

# UPDATE PORTFOLIO
# We need to adapt for a Duration D = 1 to make it stable to changes in interest rate (plan is to invest for 1 year)
# Solve to find wA and wB for building portfolio with Duration = 1
D = 1

y = np.array([1, D])
A = np.array([[1, 1],
              [Da_t2_2, Db_t2_2]])
w2_2 = np.linalg.solve(A, y)
print(f"The weights to achieve D = 1 are: wA = {round(w2_2[0], 4)}, wB = {round(w2_2[1], 4)}")

a2_2 = round(w2_2[0] * P_t2_2 / Pa_t2_2, 4)
b2_2 = round(w2_2[1] * P_t2_2 / Pb_t2_2, 4)
print(f"""The portfolio (value = ${round(Pa_t2_2 * a2_2 + Pb_t2_2 * b2_2, 2)}) is built with:
        * {a2_2} bonds of type A for a value of ${round(Pa_t2_2 * a2_2, 2)}
        * {b2_2} bonds of type B for a value of ${round(Pb_t2_2 * b2_2, 2)}
        
    Note that to adapt our portfolio at t = 1 we need to: 
        * sell {round(a1-a2_2, 4)} bonds of type A at the new price ${Pa_t2_2}
        * buy {b2_2} bonds of type B at the new price ${Pb_t2_2}""")

Bond A value after interest rate change: $110.167
Bond A duration after interest rate change: 2.749
Bond B value after interest rate change: $94.176
Bond B duration: 1
At this point the portfoltio is valued: $94406.45
The weights to achieve D = 1 are: wA = 0.0, wB = 1.0
The portfolio (value = $94406.46) is built with:
        * 0.0 bonds of type A for a value of $0.0
        * 1002.4471 bonds of type B for a value of $94406.46

    Note that to adapt our portfolio at t = 1 we need to: 
        * sell 329.55 bonds of type A at the new price $110.167
        * buy 1002.4471 bonds of type B at the new price $94.176


Because the duration left in our investment is exactly 1 year, we sell all bonds of type A and only purchase bonds of type B with our money

This will secure a portfolio with D = 1 to fit our strategy

The final value will be given by the maturity of bonds of type B: $100 * 1002.4471 = $100,244.71

### FINAL REMARKS
As we can see, we end up with more than $100,000 in each scenario which is our target investment after 3 years

We consistently adapted our bond portfolio to minimize variations to our target investment over time as the interest rates were changing